# Stage 1: Multi-Quarter 13F-HR Filing Acquisition (100 Verified Funds)

Downloads up to 4 quarterly 13F-HR filings per fund for 100 verified
institutional investors from SEC EDGAR. Uses **batch processing** (10 per batch)
with 1.5 s per-request throttle for SEC fair-access compliance. All quantitative
/ algorithmic funds have been excluded from the registry.

Every CIK is classified into one of four states:
| State | Meaning |
|---|---|
| `SUCCESS` | Filing downloaded **and** dated 2024–2026 |
| `STALE` | Filing found but date falls outside the 2024–2026 window |
| `MISSING_FORM` | SEC EDGAR has no `13F-HR` on file for this CIK |
| `RATE_LIMITED` | SEC returned HTTP 403 / 429 — request was throttled |

In [1]:
# ── Set Working Directory ─────────────────────────────────────────────────────
# Ensure working directory is the Model/ folder so all relative paths
# (./data/, ./vector_db/, etc.) resolve consistently across notebooks.
import os, pathlib

_cwd = pathlib.Path.cwd()
# If VS Code opened the workspace root, navigate into Model/
if not (_cwd / "data").exists() and (_cwd / "Model" / "data").exists():
    os.chdir(_cwd / "Model")

os.makedirs("data", exist_ok=True)
print(f"Working directory: {os.getcwd()}")

Working directory: c:\Users\user\Documents\1fintech\GenAI\Individual Assignment 2\Model


In [2]:
!pip install sec-edgar-downloader tqdm

You should consider upgrading via the 'C:\Users\user\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [3]:
import os
import re
import json
import time
import logging
from sec_edgar_downloader import Downloader

# ── Logging setup ────────────────────────────────────────────────────────────
os.makedirs("./data", exist_ok=True)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler("./data/stage1_download.log", mode="w", encoding="utf-8"),
    ],
)
log = logging.getLogger("stage1")

# ── Configuration ────────────────────────────────────────────────────────────
DOWNLOAD_DIR   = "./data/13f_filings"
FILING_TYPE    = "13F-HR"
FILING_LIMIT   = 4            # filings per CIK (≈4 quarters for cross-period analysis)
BATCH_SIZE     = 10           # funds per batch
REQUEST_DELAY  = 1.5          # seconds between individual downloads (SEC fair-access)
VALID_YEARS    = {"2024", "2025", "2026"}   # expanded for multi-period coverage

# ── User-Agent verification ──────────────────────────────────────────────────
UA_NAME  = "13F-Insight Research"
UA_EMAIL = "your-email@example.com"
log.info(f"User-Agent → company_name='{UA_NAME}', email='{UA_EMAIL}'")
print(f"User-Agent header: {UA_NAME} / {UA_EMAIL}")

# ══════════════════════════════════════════════════════════════════════════════
#  CIK REGISTRY — 100 Verified 13F-HR Filers
#  Every CIK below has been validated against SEC EDGAR to confirm it
#  actually files Form 13F-HR with recent (2024-2026) submissions.
# ══════════════════════════════════════════════════════════════════════════════

FUNDS = {
    # ═══ 1. ICONIC / LEGENDARY INVESTORS (3) ════════════════════════════════
    "0001067983": "Berkshire Hathaway Inc (Warren Buffett)",
    "0001649339": "Scion Asset Management (Michael Burry)",
    "0001350694": "Bridgewater Associates (Ray Dalio)",

    # ═══ 2. MEGA-CAP ASSET MANAGERS (8) ═════════════════════════════════════
    "0001166559": "Vanguard Group Inc",
    "0001336528": "Fidelity Management & Research (FMR)",
    "0001214717": "Geode Capital Management",
    "0000073124": "Northern Trust Corp",
    "0001126328": "Ariel Investments",
    "0000764038": "Dimensional Fund Advisors",
    "0001390777": "BNY Mellon Investment Management",
    "0001224962": "WCM Investment Management",

    # ═══ 3. MAJOR BANKS & FINANCIAL SERVICES (12) ═══════════════════════════
    "0000102909": "Morgan Stanley",
    "0000895421": "Goldman Sachs Group",
    "0000070858": "JPMorgan Chase & Co",
    "0000019617": "Bank of America Corp",
    "0000093751": "Wells Fargo & Company",
    "0000886982": "UBS Group AG",
    "0000831001": "Citigroup Inc",
    "0000312069": "Barclays PLC",
    "0000072971": "Ameriprise Financial Inc",
    "0000720005": "Raymond James Financial Inc",
    "0000036104": "US Bancorp",
    "0001137774": "RBC Global Asset Management",

    # ═══ 4. LARGE HEDGE FUNDS — NON-QUANT (15) ═════════════════════════════
    "0001035674": "Third Point LLC (Dan Loeb)",
    "0001103804": "Baupost Group (Seth Klarman)",
    "0001276520": "Tiger Global Management",
    "0001418814": "Whale Rock Capital",
    "0001061768": "Lone Pine Capital",
    "0001040273": "Maverick Capital",
    "0000909661": "Farallon Capital Management",
    "0001390113": "Taconic Capital Advisors",
    "0001535392": "Dragoneer Investment Group",
    "0001345471": "Och-Ziff Capital Management",
    "0001167557": "Cerberus Capital Management",
    "0001762304": "Hillhouse Capital (HHLR Advisors)",
    "0001536029": "Advance Capital Management",
    "0001357955": "Cevian Capital",
    "0001802994": "D1 Capital Partners",

    # ═══ 5. ACTIVIST INVESTORS (3) ══════════════════════════════════════════
    "0001308668": "ValueAct Capital",
    "0001549575": "Mantle Ridge LP",
    "0001595082": "Davidson Kempner Capital Management",

    # ═══ 6. VALUE / LONG-ONLY MANAGERS (5) ══════════════════════════════════
    "0000821197": "Johnson Investment Counsel",
    "0000853758": "Mercer Global Advisors",
    "0000938206": "Driehaus Capital Management",
    "0000740260": "Davis Selected Advisers",
    "0000859804": "Federated Hermes Inc",

    # ═══ 7. INSURANCE & FINANCIAL CONGLOMERATES (4) ═════════════════════════
    "0000004977": "AXA Financial Inc",
    "0001099219": "Lincoln National Corp",
    "0000810958": "Loews Corp",
    "0000096223": "Transamerica Corp",

    # ═══ 8. PENSION / SOVEREIGN / ENDOWMENT (7) ════════════════════════════
    "0000937567": "Ontario Teachers Pension Plan Board",
    "0001463559": "Alberta Investment Management Corp",
    "0001761054": "Treasurer of the State of North Carolina",
    "0001600177": "Employees Provident Fund Board",
    "0001603328": "Forsta AP-Fonden (Swedish Pension)",
    "0001039067": "Carnegie Mellon University",
    "0001346824": "California State Teachers Retirement",

    # ═══ 9. INTERNATIONAL ASSET MANAGERS (8) ════════════════════════════════
    "0001352924": "Caledonia (Private) Investments",
    "0001426748": "Lazard Freres Gestion S.A.S.",
    "0001629869": "Polunin Capital Partners",
    "0001686776": "Northcape Capital Pty Ltd",
    "0001313360": "SG Americas Securities (Societe Generale)",
    "0001706511": "Israel Discount Bank of New York",
    "0001872766": "FifthDelta Ltd (London)",
    "0001965104": "Manchester Global Management (UK)",

    # ═══ 10. GROWTH / TMT-FOCUSED FUNDS (1) ════════════════════════════════
    "0001517137": "Hillman Company",

    # ═══ 11. HEALTHCARE & SECTOR FUNDS (5) ══════════════════════════════════
    "0001542339": "Essex Woodlands Management",
    "0001353254": "EJF Capital LLC",
    "0001596452": "Sivik Global Healthcare LLC",
    "0001281761": "Baker Brothers Advisors",
    "0001108969": "Chatham Capital Group",

    # ═══ 12. MULTI-STRATEGY / EVENT-DRIVEN (3) ═════════════════════════════
    "0001334199": "Cambridge Financial Group",
    "0001279708": "Silver Point Capital",
    "0001039565": "ClearBridge Investments",

    # ═══ 13. REGIONAL / BOUTIQUE MANAGERS (19) ═════════════════════════════
    "0001302842": "Merk Investments LLC",
    "0001507115": "Madison Investment Advisors",
    "0001053994": "Sanders Morris Harris LLC",
    "0001005441": "Avantax Planning Partners",
    "0001395055": "Callahan Advisors LLC",
    "0001675762": "Gilbert & Cook Inc",
    "0001352547": "Legacy Private Trust Co",
    "0001730546": "Luken Investment Analytics",
    "0001318055": "Farmers Trust Co",
    "0001102256": "Towne Trust Company NA",
    "0001847772": "EMC Capital Management",
    "0001881335": "Dominguez Wealth Management Solutions",
    "0001908186": "Missouri Trust & Investment Co",
    "0001803662": "Magnolia Capital Advisors",
    "0001790837": "Bull Street Advisors LLC",
    "0001811491": "Auxano Advisors LLC",
    "0001632078": "Spectrum Asset Management (NB/CA)",
    "0001513038": "Perkins Coie Trust Co",
    "0001943228": "Horizons Wealth Management",

    # ═══ 14. ADDITIONAL INSTITUTIONAL INVESTORS (7) ═════════════════════════
    "0001703383": "Pinnacle Bancorp Inc",
    "0000764611": "Arkansas Financial Group Inc",
    "0001046187": "Sands Capital Management",
    "0001632187": "Community Bank N.A.",
    "0001869685": "Mirabaud & Cie SA",
    "0001927543": "Atria Wealth Solutions",
    "0001849529": "Revere Asset Management",
}

log.info(f"CIK registry loaded: {len(FUNDS)} funds targeted")
print(f"Registry: {len(FUNDS)} funds | Filing limit: {FILING_LIMIT} per CIK | "
      f"Batch size: {BATCH_SIZE} | Request delay: {REQUEST_DELAY}s")

17:46:01  INFO      User-Agent → company_name='13F-Insight Research', email='your-email@example.com'
17:46:01  INFO      CIK registry loaded: 100 funds targeted


User-Agent header: 13F-Insight Research / your-email@example.com
Registry: 100 funds | Filing limit: 4 per CIK | Batch size: 10 | Request delay: 1.5s


In [4]:
from tqdm import tqdm
from urllib.error import HTTPError

# ── Initialise SEC downloader ────────────────────────────────────────────────
dl = Downloader(
    company_name=UA_NAME,
    email_address=UA_EMAIL,
    download_folder=DOWNLOAD_DIR,
)

# ── Helper: extract filing year from SGML header ────────────────────────────
def _filing_year(submission_path: str) -> str:
    """Return the 4-digit year from 'FILED AS OF DATE' or 'UNKNOWN'."""
    try:
        with open(submission_path, "r", encoding="utf-8", errors="replace") as f:
            header = f.read(4096)
        m = re.search(r"FILED AS OF DATE:\s*(\d{8})", header)
        return m.group(1)[:4] if m else "UNKNOWN"
    except OSError:
        return "UNKNOWN"


# ── Retry configuration ─────────────────────────────────────────────────────
MAX_RETRIES       = 3         # retries per CIK before marking as failed
BASE_BACKOFF      = 5.0       # initial backoff seconds on 403/429
RETRY_PASS_DELAY  = 3.0       # seconds between requests in the retry pass


def _download_one_cik(cik, fund_name, attempt_delay=REQUEST_DELAY):
    """Download up to FILING_LIMIT filings for a CIK. Returns a result dict."""
    record = {"cik": cik, "fund_name": fund_name, "status": None, "detail": "", "filings": 0}

    # ── Skip if already have enough filings on disk ──────────────────
    fund_dir = os.path.join(DOWNLOAD_DIR, "sec-edgar-filings", cik, FILING_TYPE)
    if os.path.isdir(fund_dir):
        accession_dirs = sorted(
            [d for d in os.listdir(fund_dir) if os.path.isdir(os.path.join(fund_dir, d))],
            reverse=True,
        )
        if len(accession_dirs) >= FILING_LIMIT:
            # Validate most-recent filing year
            sub_path = os.path.join(fund_dir, accession_dirs[0], "full-submission.txt")
            if os.path.isfile(sub_path):
                year = _filing_year(sub_path)
                if year in VALID_YEARS:
                    record["status"] = "SUCCESS"
                    record["filings"] = len(accession_dirs)
                    record["detail"] = f"{len(accession_dirs)} filings on disk (latest {year})"
                    log.info(f"[CACHED]        CIK={cik}  {fund_name}  — {record['detail']}")
                    return record
                else:
                    record["status"] = "STALE"
                    record["filings"] = len(accession_dirs)
                    record["detail"] = f"Latest filing {year} — outside valid window (on disk)"
                    log.warning(f"[STALE]         CIK={cik}  {fund_name}  — {record['detail']}")
                    return record
        elif accession_dirs:
            # Have some filings but fewer than FILING_LIMIT — re-download to get more
            log.info(f"[PARTIAL]       CIK={cik}  {fund_name}  — {len(accession_dirs)}/{FILING_LIMIT} on disk, re-fetching")

    # ── Attempt download with retries ────────────────────────────────
    last_error = ""
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            dl.get(FILING_TYPE, cik, limit=FILING_LIMIT)
            break  # download succeeded
        except Exception as exc:
            err_msg = str(exc)
            is_rate_limit = (
                (isinstance(exc, HTTPError) and exc.code in (403, 429))
                or "403" in err_msg or "429" in err_msg
            )
            last_error = err_msg

            if is_rate_limit and attempt < MAX_RETRIES:
                wait = BASE_BACKOFF * (2 ** (attempt - 1))
                log.warning(f"[RETRY {attempt}/{MAX_RETRIES}] CIK={cik}  {fund_name}  "
                            f"— rate limited, waiting {wait:.0f}s")
                time.sleep(wait)
                continue
            elif is_rate_limit:
                record["status"] = "RATE_LIMITED"
                record["detail"] = f"HTTP 403/429 after {MAX_RETRIES} attempts"
                log.warning(f"[RATE_LIMITED]  CIK={cik}  {fund_name}  — {record['detail']}")
                time.sleep(attempt_delay)
                return record
            else:
                record["status"] = "MISSING_FORM"
                record["detail"] = err_msg
                log.warning(f"[MISSING_FORM]  CIK={cik}  {fund_name}  — {err_msg}")
                time.sleep(attempt_delay)
                return record

    # ── Locate downloaded submissions ────────────────────────────────
    fund_dir = os.path.join(DOWNLOAD_DIR, "sec-edgar-filings", cik, FILING_TYPE)
    if not os.path.isdir(fund_dir):
        record["status"] = "MISSING_FORM"
        record["detail"] = "Download returned OK but no directory created"
        log.warning(f"[MISSING_FORM]  CIK={cik}  {fund_name}  — {record['detail']}")
        time.sleep(attempt_delay)
        return record

    accession_dirs = sorted(
        [d for d in os.listdir(fund_dir) if os.path.isdir(os.path.join(fund_dir, d))],
        reverse=True,
    )

    if not accession_dirs:
        record["status"] = "MISSING_FORM"
        record["detail"] = "No accession directories found after download"
        log.warning(f"[MISSING_FORM]  CIK={cik}  {fund_name}  — {record['detail']}")
        time.sleep(attempt_delay)
        return record

    record["filings"] = len(accession_dirs)

    # ── Validate most-recent filing year ─────────────────────────────
    sub_path = os.path.join(fund_dir, accession_dirs[0], "full-submission.txt")
    if not os.path.isfile(sub_path):
        record["status"] = "MISSING_FORM"
        record["detail"] = "No full-submission.txt in newest accession directory"
        log.warning(f"[MISSING_FORM]  CIK={cik}  {fund_name}  — {record['detail']}")
        time.sleep(attempt_delay)
        return record

    year = _filing_year(sub_path)
    if year in VALID_YEARS:
        record["status"] = "SUCCESS"
        record["detail"] = f"{len(accession_dirs)} filing(s), latest {year}"
        log.info(f"[SUCCESS]       CIK={cik}  {fund_name}  — {record['detail']}")
    else:
        record["status"] = "STALE"
        record["detail"] = f"{len(accession_dirs)} filing(s), latest {year} — outside valid window"
        log.warning(f"[STALE]         CIK={cik}  {fund_name}  — {record['detail']}")

    time.sleep(attempt_delay)
    return record


# ══════════════════════════════════════════════════════════════════════════════
#  PASS 1: Main acquisition loop
# ══════════════════════════════════════════════════════════════════════════════
results = {}          # cik → result dict (deduplicates across passes)
valid_ciks = {}       # cik → fund_name  (only SUCCESS entries)

fund_items = list(FUNDS.items())
batches = [fund_items[i:i + BATCH_SIZE] for i in range(0, len(fund_items), BATCH_SIZE)]

log.info(f"PASS 1: Starting acquisition of {len(fund_items)} funds in {len(batches)} batch(es)")
print(f"\n{'='*72}")
print(f"PASS 1 — Downloading {len(fund_items)} funds × {FILING_LIMIT} filings each")
print(f"{'='*72}\n", flush=True)

for batch_num, batch in enumerate(batches, 1):
    print(f"--- Batch {batch_num}/{len(batches)} ---")
    for cik, fund_name in tqdm(batch, desc=f"Batch {batch_num}", leave=False):
        record = _download_one_cik(cik, fund_name, attempt_delay=REQUEST_DELAY)
        results[cik] = record
        if record["status"] == "SUCCESS":
            valid_ciks[cik] = fund_name
    log.info(f"Batch {batch_num} complete")

pass1_success = sum(1 for r in results.values() if r["status"] == "SUCCESS")
pass1_failed  = [cik for cik, r in results.items() if r["status"] == "RATE_LIMITED"]
print(f"\nPass 1 complete: {pass1_success} SUCCESS, {len(pass1_failed)} RATE_LIMITED")

# ══════════════════════════════════════════════════════════════════════════════
#  PASS 2: Retry all RATE_LIMITED CIKs with longer delay
# ══════════════════════════════════════════════════════════════════════════════
if pass1_failed:
    log.info(f"PASS 2: Retrying {len(pass1_failed)} rate-limited CIKs with {RETRY_PASS_DELAY}s delay")
    print(f"\n{'='*72}")
    print(f"PASS 2 — Retrying {len(pass1_failed)} rate-limited CIKs (delay={RETRY_PASS_DELAY}s)")
    print(f"{'='*72}\n", flush=True)

    print("Cooling down 30s before retry pass ...")
    time.sleep(30)

    for cik in tqdm(pass1_failed, desc="Retry pass"):
        fund_name = FUNDS[cik]
        record = _download_one_cik(cik, fund_name, attempt_delay=RETRY_PASS_DELAY)
        results[cik] = record
        if record["status"] == "SUCCESS":
            valid_ciks[cik] = fund_name

    pass2_recovered = sum(1 for cik in pass1_failed if results[cik]["status"] == "SUCCESS")
    print(f"\nPass 2 complete: recovered {pass2_recovered} / {len(pass1_failed)} previously failed CIKs")
else:
    print("No rate-limited CIKs — skipping retry pass.")

# ── Persist valid CIKs for downstream stages ────────────────────────────────
with open("./data/valid_ciks.json", "w", encoding="utf-8") as f:
    json.dump(valid_ciks, f, indent=2)

stale_entries = [r for r in results.values() if r["status"] == "STALE"]
with open("./data/stale_ciks.json", "w", encoding="utf-8") as f:
    json.dump(stale_entries, f, indent=2)

final_success = sum(1 for r in results.values() if r["status"] == "SUCCESS")
total_filings = sum(r.get("filings", 0) for r in results.values() if r["status"] == "SUCCESS")
print(f"\nAcquisition complete. {final_success} valid funds ({total_filings} total filings) "
      f"written to ./data/valid_ciks.json")

17:46:01  INFO      PASS 1: Starting acquisition of 100 funds in 10 batch(es)



PASS 1 — Downloading 100 funds × 4 filings each

--- Batch 1/10 ---


Batch 1:   0%|          | 0/10 [00:00<?, ?it/s]17:46:01  INFO      [CACHED]        CIK=0001067983  Berkshire Hathaway Inc (Warren Buffett)  — 4 filings on disk (latest 2026)
17:46:01  INFO      [CACHED]        CIK=0001649339  Scion Asset Management (Michael Burry)  — 4 filings on disk (latest 2025)
17:46:01  INFO      [CACHED]        CIK=0001350694  Bridgewater Associates (Ray Dalio)  — 4 filings on disk (latest 2026)
17:46:01  INFO      [CACHED]        CIK=0001166559  Vanguard Group Inc  — 4 filings on disk (latest 2026)
17:46:01  INFO      [CACHED]        CIK=0001336528  Fidelity Management & Research (FMR)  — 4 filings on disk (latest 2026)
17:46:01  INFO      [CACHED]        CIK=0001214717  Geode Capital Management  — 4 filings on disk (latest 2026)
17:46:01  INFO      [CACHED]        CIK=0000073124  Northern Trust Corp  — 4 filings on disk (latest 2025)
17:46:01  INFO      [CACHED]        CIK=0001126328  Ariel Investments  — 4 filings on disk (latest 2026)
17:46:01  INFO      [CAC

--- Batch 2/10 ---


Batch 2:   0%|          | 0/10 [00:00<?, ?it/s]17:46:01  INFO      [CACHED]        CIK=0001224962  WCM Investment Management  — 4 filings on disk (latest 2026)
17:46:01  INFO      [CACHED]        CIK=0000102909  Morgan Stanley  — 4 filings on disk (latest 2025)
17:46:01  INFO      [CACHED]        CIK=0000895421  Goldman Sachs Group  — 4 filings on disk (latest 2026)
17:46:01  INFO      [CACHED]        CIK=0000070858  JPMorgan Chase & Co  — 4 filings on disk (latest 2026)
17:46:01  INFO      [CACHED]        CIK=0000019617  Bank of America Corp  — 4 filings on disk (latest 2025)
17:46:01  INFO      [CACHED]        CIK=0000093751  Wells Fargo & Company  — 4 filings on disk (latest 2026)
17:46:01  INFO      [CACHED]        CIK=0000886982  UBS Group AG  — 4 filings on disk (latest 2026)
17:46:01  INFO      [CACHED]        CIK=0000831001  Citigroup Inc  — 4 filings on disk (latest 2026)
17:46:01  INFO      [CACHED]        CIK=0000312069  Barclays PLC  — 4 filings on disk (latest 2026)
17:46:

...

--- Batch 9/10 ---


Batch 9:   0%|          | 0/10 [00:00<?, ?it/s]17:46:05  INFO      [CACHED]        CIK=0001352547  Legacy Private Trust Co  — 4 filings on disk (latest 2026)
17:46:05  INFO      [CACHED]        CIK=0001730546  Luken Investment Analytics  — 4 filings on disk (latest 2026)
17:46:05  INFO      [CACHED]        CIK=0001318055  Farmers Trust Co  — 4 filings on disk (latest 2026)
17:46:05  INFO      [CACHED]        CIK=0001102256  Towne Trust Company NA  — 4 filings on disk (latest 2026)
17:46:05  INFO      [CACHED]        CIK=0001847772  EMC Capital Management  — 4 filings on disk (latest 2026)
17:46:05  INFO      [CACHED]        CIK=0001881335  Dominguez Wealth Management Solutions  — 4 filings on disk (latest 2026)
17:46:05  INFO      [CACHED]        CIK=0001908186  Missouri Trust & Investment Co  — 4 filings on disk (latest 2026)
17:46:05  INFO      [CACHED]        CIK=0001803662  Magnolia Capital Advisors  — 4 filings on disk (latest 2026)
17:46:05  INFO      [CACHED]        CIK=00017908

--- Batch 10/10 ---


Batch 10:   0%|          | 0/10 [00:00<?, ?it/s]17:46:05  INFO      [CACHED]        CIK=0001632078  Spectrum Asset Management (NB/CA)  — 4 filings on disk (latest 2026)
17:46:05  INFO      [CACHED]        CIK=0001513038  Perkins Coie Trust Co  — 4 filings on disk (latest 2026)
17:46:05  INFO      [CACHED]        CIK=0001943228  Horizons Wealth Management  — 4 filings on disk (latest 2026)
17:46:05  INFO      [CACHED]        CIK=0001703383  Pinnacle Bancorp Inc  — 4 filings on disk (latest 2026)
17:46:05  INFO      [CACHED]        CIK=0000764611  Arkansas Financial Group Inc  — 4 filings on disk (latest 2026)
17:46:05  INFO      [CACHED]        CIK=0001046187  Sands Capital Management  — 4 filings on disk (latest 2026)
17:46:05  INFO      [CACHED]        CIK=0001632187  Community Bank N.A.  — 4 filings on disk (latest 2026)
17:46:05  INFO      [CACHED]        CIK=0001869685  Mirabaud & Cie SA  — 4 filings on disk (latest 2026)
17:46:05  INFO      [CACHED]        CIK=0001927543  Atria We


Pass 1 complete: 100 SUCCESS, 0 RATE_LIMITED
No rate-limited CIKs — skipping retry pass.

Acquisition complete. 100 valid funds (397 total filings) written to ./data/valid_ciks.json


In [5]:
# ══════════════════════════════════════════════════════════════════════════════
#  ACQUISITION REPORT — explains why we retained N out of 100 funds
# ══════════════════════════════════════════════════════════════════════════════

from collections import Counter

# ── Status summary ───────────────────────────────────────────────────────────
all_results = list(results.values())
counts = Counter(r["status"] for r in all_results)
total  = len(all_results)

print("=" * 72)
print(f"{'ACQUISITION REPORT':^72}")
print("=" * 72)
print(f"\n{'Status':<16} {'Count':>6}  {'Pct':>6}")
print("-" * 34)
for status in ["SUCCESS", "STALE", "MISSING_FORM", "RATE_LIMITED"]:
    n = counts.get(status, 0)
    pct = f"{100 * n / total:.1f}%" if total else "—"
    print(f"  {status:<14} {n:>5}   {pct:>5}")
print("-" * 34)
print(f"  {'TOTAL':<14} {total:>5}")

# ── Filing count summary ─────────────────────────────────────────────────────
total_filings = sum(r.get("filings", 0) for r in all_results if r["status"] == "SUCCESS")
avg_filings = total_filings / max(counts.get("SUCCESS", 0), 1)
print(f"\n  Cross-period filings: {total_filings} total across {counts.get('SUCCESS', 0)} funds "
      f"(avg {avg_filings:.1f} per fund, target {FILING_LIMIT})")

# ── Detailed per-fund table ──────────────────────────────────────────────────
print(f"\n{'#':>3}  {'CIK':<12} {'Status':<14} {'Qty':>3}  {'Fund Name':<42} Detail")
print("-" * 108)
for i, r in enumerate(all_results, 1):
    qty = r.get("filings", 0)
    print(f"{i:>3}  {r['cik']:<12} {r['status']:<14} {qty:>3}  {r['fund_name']:<42} {r['detail']}")

# ── Explanation ──────────────────────────────────────────────────────────────
print(f"\n{'=' * 72}")
print(f"Summary: {counts.get('SUCCESS', 0)} funds passed all checks (SUCCESS).")
excluded = total - counts.get("SUCCESS", 0)
if excluded:
    reasons = []
    if counts.get("STALE", 0):
        reasons.append(f"{counts['STALE']} STALE (filing year outside valid window)")
    if counts.get("MISSING_FORM", 0):
        reasons.append(f"{counts['MISSING_FORM']} MISSING_FORM (no 13F-HR on EDGAR)")
    if counts.get("RATE_LIMITED", 0):
        reasons.append(f"{counts['RATE_LIMITED']} RATE_LIMITED (SEC 403/429 after retries)")
    print(f"Excluded {excluded} fund(s): " + "; ".join(reasons) + ".")
print(f"{'=' * 72}")
print(f"\nValid CIKs saved to  : ./data/valid_ciks.json  ({counts.get('SUCCESS', 0)} entries)")
print(f"Stale CIKs saved to  : ./data/stale_ciks.json   ({counts.get('STALE', 0)} entries)")
print(f"Full log              : ./data/stage1_download.log")
print(f"\nReady for Stage 2 parsing.")

                           ACQUISITION REPORT                           

Status            Count     Pct
----------------------------------
  SUCCESS          100   100.0%
  STALE              0    0.0%
  MISSING_FORM       0    0.0%
  RATE_LIMITED       0    0.0%
----------------------------------
  TOTAL            100

  Cross-period filings: 397 total across 100 funds (avg 4.0 per fund, target 4)

  #  CIK          Status         Qty  Fund Name                                  Detail
------------------------------------------------------------------------------------------------------------
  1  0001067983   SUCCESS          4  Berkshire Hathaway Inc (Warren Buffett)    4 filings on disk (latest 2026)
  2  0001649339   SUCCESS          4  Scion Asset Management (Michael Burry)     4 filings on disk (latest 2025)
  3  0001350694   SUCCESS          4  Bridgewater Associates (Ray Dalio)         4 filings on disk (latest 2026)
  4  0001166559   SUCCESS          4  Vanguard Group Inc  